# BPVAv2 三步真实数据 Smoke Test

1. 构造或加载 BPVAv2 policy；
2. 加载真实 LeRobot 数据集，BP 只读取 `image0`；
3. 生成 batch、打印 Query Compressor 中间 shape，并运行 forward / 推理。

默认关闭 `lambda_gen`、`lambda_3d`，避免 smoke test 运行辅助 target loss 和 DA3 teacher。Cosmos middle 路径仍会运行，所以 Cosmos tokenizer 路径必须有效。

In [15]:
# 第一步：构造或加载 BPVAv2 policy
import copy
import os
import random
from dataclasses import fields
from pathlib import Path

import torch

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.BPVAv2.configuration_bpva import BPVAv2Config
from lerobot.policies.BPVAv2.modeling_bpva import BPVAv2Policy
from lerobot.utils.constants import OBS_IMAGES

LOAD_MODE = "BPVAV2"  # "SCRATCH" / "TBOT" / "BPVAV2"
# CHECKPOINT_DIR = Path("/home/jovyan/workspace/models/tbot-pretrain-v2")
CHECKPOINT_DIR = Path("/home/jovyan/workspace/models/bpvas/bpvav2_init_v0.1")
QWEN3_VL_DIR = Path("/home/jovyan/workspace/models/Qwen3-vl-2b-instruct")
COSMOS_DIR = Path("/home/jovyan/workspace/models/nvidia-cosmos-tokwnizer-ci8x8")
DEVICE = "cuda:0"
DTYPE = "bfloat16"
SEED = 0
POLICY_BP_CAMERA_KEYS = [
    f"{OBS_IMAGES}.image0",
    f"{OBS_IMAGES}.image1",
    f"{OBS_IMAGES}.image2",
]
ACTIVE_BP_CAMERA_KEYS = [f"{OBS_IMAGES}.image0"]

random.seed(SEED)
torch.manual_seed(SEED)


RUNTIME_OVERRIDES = dict(
    pretrained_path=None,
    qwen3_vl_pretrained_path=str(QWEN3_VL_DIR),
    cosmos_tokenizer_path_or_name=str(COSMOS_DIR),
    device=DEVICE,
    dtype=DTYPE,
    lambda_gen=0.0,
    lambda_3d=0.0,
    gradient_checkpointing=False,
)

BPVAV2_STRUCTURE_OVERRIDES = dict(
    bp_camera_keys=list(POLICY_BP_CAMERA_KEYS),
    bp_encoder_version="query_compressor_v1",
    bp_num_chunks=10,
    bp_action_chunk_size=50,
    bp_compressor_dim=512,
    bp_state_action_hidden_dim=256,
    bp_num_query_tokens=5,
    bp_compressor_num_layers=2,
    bp_compressor_num_heads=8,
    bp_compressor_ff_mult=4,
    bp_freeze_shared_visual=True,
    bp_use_modality_type_embedding=True,
    bp_use_camera_embedding=True,
    bp_use_chunk_position_embedding=True,
)


def _copy_bpvav2_fields(source: PreTrainedConfig) -> dict:
    allowed = {field.name for field in fields(BPVAv2Config) if field.init}
    return {name: copy.deepcopy(getattr(source, name)) for name in allowed if hasattr(source, name)}


def make_config(source_dir: Path, load_mode: str) -> BPVAv2Config:
    source = PreTrainedConfig.from_pretrained(source_dir, local_files_only=True)
    values = _copy_bpvav2_fields(source)
    values.update(copy.deepcopy(RUNTIME_OVERRIDES))

    if load_mode in {"SCRATCH", "TBOT"}:
        # SCRATCH/TBOT 是“构造一个新的 BPVAv2 结构”，所以允许用 notebook 顶部结构参数覆盖。
        values.update(copy.deepcopy(BPVAV2_STRUCTURE_OVERRIDES))
    elif load_mode == "BPVAV2":
        # BPVAV2 是“恢复已有 BPVAv2 checkpoint”，结构字段必须以 checkpoint config 为准。
        if getattr(source, "type", None) != "bpvav2":
            raise ValueError(f"LOAD_MODE='BPVAV2' requires a bpvav2 checkpoint, got {getattr(source, 'type', None)!r}")
    else:
        raise ValueError(load_mode)

    return BPVAv2Config(**values)


cfg = make_config(CHECKPOINT_DIR, LOAD_MODE)
if LOAD_MODE == "SCRATCH":
    policy = BPVAv2Policy(cfg)
elif LOAD_MODE in {"TBOT", "BPVAV2"}:
    policy = BPVAv2Policy.from_pretrained(
        CHECKPOINT_DIR, config=cfg, local_files_only=True, strict=False
    )
else:
    raise ValueError(LOAD_MODE)
policy.eval()

if any(key not in cfg.bp_camera_keys for key in ACTIVE_BP_CAMERA_KEYS):
    raise ValueError(f"ACTIVE_BP_CAMERA_KEYS must be a subset of checkpoint policy slots: {cfg.bp_camera_keys}")

print("load mode/checkpoint:", LOAD_MODE, CHECKPOINT_DIR)
print("policy/device/dtype:", policy.name, cfg.device, cfg.dtype)
print("policy slots:", cfg.bp_camera_keys)
print("active BP cameras:", ACTIVE_BP_CAMERA_KEYS)
print("K/Q/compressor D:", cfg.bp_num_chunks, cfg.bp_num_query_tokens, cfg.bp_compressor_dim)


Loading weights from local directory


ValueError: BPVAv2 query-compressor checkpoint is structurally incompatible: bp_camera_keys: source=['observation.images.image0'], current=['observation.images.image0', 'observation.images.image1', 'observation.images.image2']; bp_num_query_tokens: source=8, current=5; bp_num_chunks: source=8, current=10

In [14]:
# 第一步补充：确认 Query Compressor 参数已注册
bp_keys = [key for key in policy.state_dict() if key.startswith("model.bp_obs_encoder.")]
required = ("query_tokens", "visual_projection", "state_mlp", "action_mlp", "compressor_layers", "output_projection")
missing = [name for name in required if not any(name in key for key in bp_keys)]
if missing:
    raise RuntimeError(f"Query Compressor 缺少参数：{missing}")
print("BP encoder tensors:", len(bp_keys))
print("final BP prefix token count:", cfg.bp_num_chunks * cfg.bp_num_query_tokens)

BP encoder tensors: 65
final BP prefix token count: 50


## 第二步：加载真实 LeRobot 数据集

`current_ds` 保持 TBot 当前观测所需的三路相机；`prompt_ds` 通过 `active_camera_keys` 只读取一路 BP 图像。

### 2.1 current / prompt dataset

In [ ]:
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.transforms.constants import get_feature_mapping, get_image_mapping
from lerobot.utils.constants import ACTION

DATASET_PATH = Path("/share/RoboTwin-LeRobot-v3.0/beat_block_hammer/aloha-agilex_randomized_500")
EPISODES = [0]
VIDEO_BACKEND = "pyav"

meta = LeRobotDatasetMetadata(str(DATASET_PATH))
current_delta_timestamps = resolve_delta_timestamps(cfg, meta)
feature_mapping = get_feature_mapping(meta.robot_type, meta.features)
prompt_delta_timestamps = {
    key: [step / meta.fps for step in range(cfg.bp_action_chunk_size)]
    for key in meta.features
    if key == ACTION or key in feature_mapping[ACTION]
}
image_mapping = get_image_mapping(meta.robot_type, meta.features)
canonical_to_actual = {canonical: actual for actual, canonical in image_mapping.items()}
active_prompt_camera_keys = [canonical_to_actual[key] for key in ACTIVE_BP_CAMERA_KEYS]

current_ds = LeRobotDataset(
    str(DATASET_PATH), episodes=EPISODES,
    delta_timestamps=current_delta_timestamps,
    video_backend=VIDEO_BACKEND, skip_video_file_validation=True,
)
prompt_ds = LeRobotDataset(
    str(DATASET_PATH), episodes=EPISODES,
    delta_timestamps=prompt_delta_timestamps,
    video_backend=VIDEO_BACKEND, skip_video_file_validation=True,
    active_camera_keys=active_prompt_camera_keys,
)
prompt_ds.meta.stats.update(current_ds.meta.stats)

print("dataset:", DATASET_PATH)
print("robot/fps/rows:", meta.robot_type, meta.fps, len(current_ds))
print("current cameras:", current_ds.meta.camera_keys)
print("active prompt cameras:", prompt_ds.active_visual_keys)
print("canonical mapping:", canonical_to_actual)


In [ ]:
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptConfig, BehaviorPromptLeRobotDataset

bp_config = BehaviorPromptConfig(
    prompt_action_chunk_size=cfg.bp_action_chunk_size,
    num_chunks=cfg.bp_num_chunks,
    same_episode_policy="avoid",
    seed=SEED,
    height=cfg.image_resolution[0],
    width=cfg.image_resolution[1],
    max_state_dim=cfg.max_state_dim,
    max_action_dim=cfg.max_action_dim,
    qwen3_vl_processor_path=cfg.qwen3_vl_pretrained_path,
    bp_camera_keys=list(ACTIVE_BP_CAMERA_KEYS),
    action_mode="delta",
    batch_prompt_video_decode=True,
    dynamic_bp_cameras=True,
)
bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms(current_ds, prompt_ds, bp_config)

SAMPLE_INDEX = random.randrange(len(bp_ds))
sample = bp_ds[SAMPLE_INDEX]
print("sample index:", SAMPLE_INDEX)
for i, transform in enumerate(bp_ds.transform.transforms):
    print(f"transform[{i:02d}] {transform.__class__.__name__}")
prompt = sample["behavior_prompt"]
print("BP state/action:", prompt["state"].shape, prompt["action"].shape)
print("BP active processed keys:", list(prompt["bp_pixel_values"]))
for key in prompt["bp_pixel_values"]:
    print(key, "pixels/grid:", prompt["bp_pixel_values"][key].shape, prompt["bp_image_grid_thw"][key].shape)

In [ ]:
# 第三步：生成 batch，并安装 BP Query Compressor shape hooks
from torch.utils.data._utils.collate import default_collate

batch = default_collate([sample])


def move_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move_to_device(item, device) for key, item in value.items()}
    if isinstance(value, list):
        return [move_to_device(item, device) for item in value]
    if isinstance(value, tuple):
        return tuple(move_to_device(item, device) for item in value)
    return value


batch_for_forward = move_to_device(batch, DEVICE)
encoder = policy.model.bp_obs_encoder.chunk_encoder
shape_log = {}
handles = []


def module_shape_hook(name):
    def hook(_module, inputs, output):
        def shape(value):
            if isinstance(value, torch.Tensor):
                return tuple(value.shape)
            if isinstance(value, tuple):
                return [shape(item) for item in value]
            return type(value).__name__
        shape_log[name] = {"in": shape(inputs), "out": shape(output)}
    return hook


for name in ("visual_projection", "state_mlp", "action_mlp", "output_projection"):
    handles.append(getattr(encoder, name).register_forward_hook(module_shape_hook(name)))
for index, layer in enumerate(encoder.compressor_layers):
    handles.append(layer.register_forward_hook(module_shape_hook(f"compressor_layer_{index}")))

print("batch current image:", batch_for_forward[f"{OBS_IMAGES}.image0"].shape)
print("batch state/action:", batch_for_forward["observation.state"].shape, batch_for_forward["action"].shape)
print("BP policy slots / active:", cfg.bp_camera_keys, list(batch_for_forward["behavior_prompt"]["bp_pixel_values"]))

In [ ]:
# 运行一次真实 forward，并打印 BP 中间 shape。
torch.cuda.empty_cache()
policy.eval()
autocast_enabled = str(DEVICE).startswith("cuda") and DTYPE == "bfloat16"
with torch.inference_mode(), torch.autocast(
    device_type="cuda", dtype=torch.bfloat16, enabled=autocast_enabled
):
    loss, loss_dict = policy(batch_for_forward)

for handle in handles:
    handle.remove()

print("forward ok, loss:", float(loss.cpu()))
for key, value in loss_dict.items():
    if not key.startswith("loss_action_dim"):
        print(f"{key}: {value}")
print("\nBP Query Compressor module shapes:")
for name, record in shape_log.items():
    print(name, record)

# 直接读取 encoder 的结构常量，便于对照流程图。
active_grid = next(iter(batch_for_forward["behavior_prompt"]["bp_image_grid_thw"].values()))
merge = int(policy.model.qwen3_vl_with_expert.und_expert.visual.spatial_merge_size)
qwen_tokens_per_image = int(active_grid[0, 0].prod().item() // (merge ** 2))
C = len(cfg.bp_camera_keys)
T = cfg.bp_action_chunk_size
Q = cfg.bp_num_query_tokens
print("\nResolved BP shapes:")
print("Qwen tokens/image N:", qwen_tokens_per_image)
print("visual memory:", (1, cfg.bp_num_chunks, C * qwen_tokens_per_image, cfg.bp_compressor_dim))
print("state memory:", (1, cfg.bp_num_chunks, 1, cfg.bp_compressor_dim))
print("action memory:", (1, cfg.bp_num_chunks, T, cfg.bp_compressor_dim))
print("total memory:", (1, cfg.bp_num_chunks, C * qwen_tokens_per_image + 1 + T, cfg.bp_compressor_dim))
print("queries before output projection:", (1 * cfg.bp_num_chunks, Q, cfg.bp_compressor_dim))
print("BP prefix after K×Q flatten:", (1, cfg.bp_num_chunks * Q, 2048))

# 可选推理：默认关闭，避免额外执行 num_inference_steps 次动作去噪。
RUN_INFERENCE = False
if RUN_INFERENCE:
    with torch.inference_mode(), torch.autocast(
        device_type="cuda", dtype=torch.bfloat16, enabled=autocast_enabled
    ):
        actions, _ = policy.predict_action_chunk(batch_for_forward)
    print("inference action chunk:", actions.shape, "finite:", bool(torch.isfinite(actions).all()))

# 探索区域

In [ ]:
policy.config

BPVAv2Config(n_obs_steps=1, input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda:0', use_amp=False, push_to_hub=False, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l', qwen3_vl_pretrained_path='/home/jovyan/workspace/models/Qwen3-vl-2b-instruct', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'STATE': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'ACTION': <NormalizationMode.IDENTITY: 'IDENTITY'>}, gradient_checkpointing=False, compile_model=False, compile_mode='max-autotune', optimizer_lr=5e-05, optimizer_betas=(0.9, 0.95), optimizer_eps=1e-08, optimizer_weight_decay=0.01, optimizer_grad_clip_norm=1.0, scheduler_warmup_steps=2000, scheduler_decay_steps=300000, scheduler_decay_lr=5e-05, tokenizer_max_length=96, freeze_vision_encoder=False, train_expert_only=False, train_vlm_only=False, lora_modules=(), lora_unselected_mode='full', lora_targets=('attn', 'ffn'), lora_rank=16, lora_alpha=32.0, lora_rank_und=None, lora_alpha_und=None, lora_rank_gen=None, lora_alpha_gen=None, lora_rank_act=None, lora_alpha_act=None, lora_dropout=0.0, scale_factor=8, lambda_gen=0.0, cosmos_tokenizer_path_or_name='/home/jovyan/workspace/models/nvidia-cosmos-tokwnizer-ci8x8', enable_3d_queries=True, num_3d_query_tokens=432, da3_alignment_mode='query_decoder', da3_query_resampler_layers=1, da3_query_resampler_ff_mult=1, query_layer_indices=(13, 19, 23, 27), da3_variant='auto', da3_teacher_layers=(11, 15, 19, 23), da3_query_dim=2048, da3_tokens_per_view=1296, da3_num_views=3, lambda_3d=0.0, da3_model_path_or_name='/home/jovyan/workspace/models/DA3-LARGE-1.1', da3_model_name=None, da3_code_root='/home/jovyan/workspace/mytbot/third_party/Depth-Anything-3', da3_teacher_process_res=504, da3_layer_weights=(1.0, 1.2, 1.4, 1.6), future_query_init_std=0.02, log_da3_teacher_timing=True, bp_num_chunks=10, bp_action_chunk_size=50, bp_camera_keys=['observation.images.image0', 'observation.images.image1', 'observation.images.image2'], bp_encoder_version='query_compressor_v1', bp_freeze_shared_visual=True, bp_compressor_dim=512, bp_state_action_hidden_dim=256, bp_num_query_tokens=5, bp_compressor_num_layers=2, bp_compressor_num_heads=8, bp_compressor_ff_mult=4, bp_use_modality_type_embedding=True, bp_use_camera_embedding=True, bp_use_chunk_position_embedding=True)